# Inscopix Experiments to NWB Format

In [ ]:
from datetime import datetime
from os.path import exists
from pathlib import Path

import pandas as pd

from tqdm.notebook import tqdm
from neuroconv.datainterfaces import InscopixImagingInterface
import isx

## 1. Path Transversal and File Loading

In [ ]:
data_dir = Path("/mnt/r2d2/2_Inscopix/1_DTT/")
output_dir = Path("/mnt/r2d2/2_Inscopix/NWB_Files")
prefixes = ["1", "2", "3", "4"]
genotypes = ["VGAT", "VGLUT", "OXTR"]
experiments = ["1_OdorAnalysis",  "2_HFvFM", "3_EPM", "4_Social"]
odor_experiment_subtypes = ["1_Concentration", "2_Identity", "4_Social_Odor"]

animal_age_path = Path("/mnt/r2d2/2_Inscopix/1_DTT/vgat_vglut_ages.xlsx")

### 1a Get Animal Ages

In [ ]:
def get_ages(row) -> str:
    age_in_days = (row["Imaged Date"] - row["DOB"]).days
    age_rounded = age_in_days - (age_in_days % 7)
    return f'P{age_rounded}D'
ages = pd.read_excel(animal_age_path, index_col=0)
ages["Age"] = ages.apply(get_ages, axis=1)

### 1b Load and save experiment folders and metadata

In [ ]:
path_dict = {}

def get_animals_per_genotypes(data_dir: Path, subtype_label: str) -> list:
    experiment_animals = []

    if "Social" in subtype_label:
        _genotypes = ["OXTR"]
    else:
        _genotypes = genotypes

    for genotype in _genotypes:
        genotype_dir = data_dir / genotype

        for animal in genotype_dir.glob(f"{genotype}*/"):
            if not animal.is_dir() or "z" in animal.name.lower():
                continue

            animal_dict = {
                "name": animal.name,
                "genotype": genotype,
                "subtype": subtype_label,
                "path": animal
            }

            experiment_animals.append(animal_dict)

    return experiment_animals

experiment_dirs = [item for item in data_dir.iterdir() if item.is_dir() and item.name[0] in prefixes]


for experiment in experiments:
    experiment_dir = data_dir / experiment
    experiment_animals = []
    if experiment == "1_OdorAnalysis":
        # Odor is special; it has subtypes
        for subtype in odor_experiment_subtypes:
            # Cycle Each Subtype
            subtype_dir = experiment_dir / subtype
            subtype_label = subtype.split("_")[1]
            experiment_animals.extend(get_animals_per_genotypes(subtype_dir, subtype_label))

        path_dict[experiment] = experiment_animals

    else:
        subtype_label = experiment.split("_")[1]
        path_dict[experiment] = get_animals_per_genotypes(experiment_dir, subtype_label)

## 2. Process and Convert Directories

In [ ]:
def get_animal_paths(animal_path: Path)-> tuple[Path, Path]:
    animal_raw_data_dir = animal_path / "RawData"
    output_path_stem = Path(*animal_path.parts[-3:])
    return animal_raw_data_dir, output_path_stem

for experiment in experiments:
    print(f"Processing {experiment}")
    experiment_output_dir = output_dir.joinpath(experiment)

    experiment_animals = path_dict[experiment]
    for animal_dict in experiment_animals:
        # Get raw data dir and stem for output dir
        raw_data_dir, output_stem = get_animal_paths(animal_dict["path"])
        animal_output_dir = experiment_output_dir.joinpath(output_stem)
        # Create output dir tree
        animal_output_dir.mkdir(parents=True, exist_ok=True)